# REES46 — Clean Temporal Inactivity Experiment (run ON KAGGLE)

This notebook runs the **Phase-7 sanity probe** followed by the **full REES46 temporal experiment** on the raw event-level `mkechinov/ecommerce-behavior-data-from-multi-category-store` dataset.

**You MUST attach this data as a Kaggle input** so the monthly files land under:
`/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/*.csv`

Attach these inputs **before** hitting Run All:
1. `mkechinov/ecommerce-behavior-data-from-multi-category-store` (the 7 monthly CSVs `2019-Oct.csv` .. `2020-Apr.csv`)

**What this does (in order):**
1. pulls the code from GitHub (adds `src/` to the path),
2. **Sanity probe** — checks the raw data on disk: monthly files, event-type coverage, event `event_time` parsing, data span vs the 90-day label window, global temporal cutoffs, real churn rates at both cutoffs, and runs the hard temporal-validity assertions on a sampled feature matrix,
3. **Full experiment** — Original + SMOTE × 5 models (LR, SVM, RF, XGB, LGBM) for REES46.

> Assessment gate: if the sanity probe reports extreme churn (~0% or ~100%) or truncated label windows at either cutoff, **stop** and review before running the full sweep.

All outputs are written under `results/rees46/{original,smote}/` (gitignored locally, but present in `/kaggle/working`). Download `/kaggle/working` when done.

In [ ]:
# ── Settings ──────────────────────────────────────────────────────────────
import os

REPO_URL = "https://github.com/nikhilwankhedee/churn.git"
REPO_BRANCH = "main"
UPDATE_REPO = True            # git pull the latest code on every run

# REES46-only sweep — the one dataset this notebook evaluates.
MAIN_TARGETS = ["rees46"]

CHURN_WINDOW_OVERRIDE = None  # None -> adapter default (90 days, temporal inactivity)
SENSITIVITY = False

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK_DIR = "/kaggle/working" if ON_KAGGLE else ("/content" if os.path.isdir("/content") else os.getcwd())
REPO_DIR = os.path.join(WORK_DIR, "churn")
print(f"Environment: {'KAGGLE' if ON_KAGGLE else 'LOCAL'} | code -> {REPO_DIR}")


In [ ]:
# ── Clone / update the code from GitHub ───────────────────────────────────────
import io, os, sys, shutil, subprocess, zipfile, urllib.request

def git(cmd, cwd=None):
    print("$ git " + " ".join(cmd))
    subprocess.check_call(["git"] + cmd, cwd=cwd)

def sync_repo(repo_dir, url, branch, update=True):
    repo_dir = os.path.abspath(repo_dir)
    parent = os.path.dirname(repo_dir)
    if os.path.isdir(os.path.join(repo_dir, ".git")):
        if not update:
            print(f"-> Using existing repo at {repo_dir} (UPDATE_REPO=False)")
        else:
            try:
                git(["fetch", "origin"], repo_dir)
                git(["checkout", "--quiet", branch], repo_dir)
                git(["pull", "--rebase", "origin", branch], repo_dir)
                print("-> Repo updated to latest.")
            except Exception as exc:
                print(f"-> Update failed ({exc}); reusing existing checkout.")
        return repo_dir
    try:
        git(["clone", "--branch", branch, url, repo_dir])
        print(f"-> Cloned {url} -> {repo_dir}")
    except Exception:
        print("git clone failed - downloading GitHub zipball instead...")
        if os.path.isdir(repo_dir):
            shutil.rmtree(repo_dir)
        os.makedirs(parent, exist_ok=True)
        zip_url = url.replace(".git", "") + f"/archive/refs/heads/{branch}.zip"
        with urllib.request.urlopen(zip_url, timeout=120) as resp:
            data = resp.read()
        extract_dir = os.path.join(parent, "__churn_extract__")
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            zf.extractall(extract_dir)
        extracted = [os.path.join(extract_dir, d) for d in os.listdir(extract_dir)
                     if os.path.isdir(os.path.join(extract_dir, d))]
        shutil.move(extracted[0], repo_dir)
        shutil.rmtree(extract_dir)
        print(f"-> Downloaded {zip_url} -> {repo_dir}")
    return repo_dir

REPO_DIR = sync_repo(REPO_DIR, REPO_URL, REPO_BRANCH, update=UPDATE_REPO)


In [ ]:
# ── Install missing dependencies ──────────────────────────────────────────────
from importlib.util import find_spec
MODULES = {"pandas":"pandas","numpy":"numpy","scikit-learn":"sklearn","xgboost":"xgboost","lightgbm":"lightgbm","imbalanced-learn":"imblearn","shap":"shap","matplotlib":"matplotlib","seaborn":"seaborn","scipy":"scipy","pingouin":"pingouin","statsmodels":"statsmodels","joblib":"joblib","tqdm":"tqdm","pyyaml":"yaml","typer":"typer","rich":"rich","openpyxl":"openpyxl"}
SKIP = {"jupyter", "ipykernel"}
missing = []
with open(os.path.join(REPO_DIR, "requirements.txt")) as fh:
    for line in fh:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        name = line.split(">=")[0].split("==")[0].split("[")[0].strip()
        if name.lower() in SKIP:
            continue
        mod = MODULES.get(name.lower(), name.lower().replace("-", "_"))
        if find_spec(mod) is None:
            missing.append(name)
if missing:
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *missing]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        print("pip refused - retrying with --break-system-packages")
        subprocess.check_call(cmd + ["--break-system-packages"])
else:
    print("All pipeline dependencies are already installed.")


In [ ]:
# ── Add code to path and verify framework + data input ────────────────
os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = REPO_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

from src.config import ON_KAGGLE, PROJECT_ROOT, REES46_MULTICATEGORY_DIR, REES46_MULTICATEGORY_FILES
from src.datasets import list_datasets, get_dataset

print(f"On Kaggle    : {ON_KAGGLE}")
print(f"Project root : {PROJECT_ROOT}")
print(f"Reg datasets : {', '.join(list_datasets())}")

# Confirm the raw multi-category input is actually attached.
present = [f for f in REES46_MULTICATEGORY_FILES
           if os.path.isfile(os.path.join(REES46_MULTICATEGORY_DIR, f))]
missing = [f for f in REES46_MULTICATEGORY_FILES if f not in present]
print(f"\nREES46 multi-category input: {REES46_MULTICATEGORY_DIR}")
print(f"  present: {present}")
print(f"  missing: {missing}")
if not present:
    raise RuntimeError(
        "No monthly CSVs attached. Add input "
        "mkechinov/ecommerce-behavior-data-from-multi-category-store "
        "so the files land under /kaggle/input/datasets/mkechinov/...")

adapter = get_dataset("rees46")
print(f"Churn window  : {adapter.churn_window_days} days (temporal inactivity)")
print(f"Native label  : {adapter.uses_native_churn_label} (must be False for the clean temporal path)")
assert not adapter.uses_native_churn_label

try:
    rev = subprocess.check_output(["git", "-C", REPO_DIR, "log", "-1", "--format=%h  %cs  %s"], text=True).strip()
    print(f"Code version : {rev}")
except Exception:
    pass


## Phase 7 — Sanity probe (the real data, before any model is fit)

This is the **gatekeeper**. It does NOT fit models. It verifies the raw data supports a valid 90-day temporal inactivity experiment: expected monthly files, event-type coverage, `event_time` parsing, data span vs the 90-day label window, global temporal cutoffs, and **real churn rates** at both cutoffs.

- If either `train_label_window_observed` / `test_label_window_observed` is **False**, the label windows are truncated — **stop**: the inactivity definition would be biased.
- If train or test churn is ~0% or ~100% (or the window is not fully observed), review the churn geometry before running the full sweep.
- The hard temporal-validity assertions (causality, no target-derived columns, label windows observed, binary labels) must all PASS.


In [ ]:
# ── Run the REES46 temporal sanity probe ───────────────────────────────────────
import json
from src.rees46_sanity_probe import sanity_probe

report = sanity_probe()

# Persist the probe report for the audit trail.
out_dir = os.path.join(PROJECT_ROOT, "results", "rees46", "preflight")
os.makedirs(out_dir, exist_ok=True)
probe_path = os.path.join(out_dir, "sanity_probe.json")
with open(probe_path, "w") as fh:
    json.dump(report, fh, indent=2, default=str)
print(f"\nSanity probe report saved: {probe_path}")

# Hard gate: fail fast on truncated label windows or missing event types.
ok_label = report["train_label_window_observed"] and report["test_label_window_observed"]
if not ok_label:
    raise RuntimeError("Label window NOT fully observed — 90-day inactivity labels would be truncated. STOP.")

ct, ck = report["churn_rate_train"], report["churn_rate_test"]
print(f"train churn {ct*100:.2f}% | test churn {ck*100:.2f}%")
if ct <= 0.01 or ct >= 0.99 or ck <= 0.01 or ck >= 0.99:
    print("WARNING: extreme churn rate — review label geometry before trusting AUC.")
else:
    print("Churn geometry looks healthy — proceeding to the full sweep.")


## Full REES46 experiment — Original + SMOTE × 5 models

Runs `run_pipeline(dataset="rees46", use_smote=False)` then `use_smote=True`. Each writes under `results/rees46/{original,smote}/`. SMOTE is applied by the pipeline strictly to the *training* fold (post split + post feature engineering) — the test set is never resampled.

> Note: the sweep code is pulled live from GitHub `main`. The temporal REES46 adapter and the sanity probe must already be merged there, or this notebook uses whatever version is currently on `main`.


In [ ]:
# ── RUN THE FULL REES46 SWEEP: original -> smote, 5 models each ─────
import time, glob, shutil
import pandas as pd
from IPython.display import display
from src.pipeline import run_pipeline

CHECKPOINT_DIR = os.path.join("/kaggle/working", "rees46_checkpoints")
BACKUP_DIR = os.path.join("/kaggle/working", "rees46_backup")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)

def _marker(name, smote):
    return os.path.join(CHECKPOINT_DIR, f"done_{name}_{'smote' if smote else 'original'}.marker")

def _is_done(name, smote):
    return os.path.exists(_marker(name, smote))

def _backup_zip(name, smote):
    mode = "smote" if smote else "original"
    root = "/kaggle/working"
    zip_name = os.path.join(BACKUP_DIR, f"{name}_{mode}.zip")
    try:
        with shutil.ZipFile(zip_name, "w", allowZip64=True) as zf:
            for top in ("results", "figures", "models", "processed_data"):
                base = os.path.join(root, top, name, mode)
                if os.path.isdir(base):
                    for fp in glob.glob(os.path.join(base, "**", "*"), recursive=True):
                        if os.path.isfile(fp):
                            zf.write(fp, os.path.relpath(fp, root))
        print(f"  -> checkpoint zip: {zip_name}")
        return zip_name
    except Exception as exc:
        print(f"  -> backup zip FAILED: {exc}")
        return None

def _run_one(name, smote):
    return run_pipeline(dataset=name, sensitivity=SENSITIVITY,
                        churn_window_override=CHURN_WINDOW_OVERRIDE, use_smote=smote)

sweep_start = time.time()
rows = []
for smote in (False, True):
    mode = "smote" if smote else "original"
    print("=" * 74)
    print(f"PHASE: {mode.upper()}  ({len(MAIN_TARGETS)} datasets x 5 models)")
    print("=" * 74)
    for name in MAIN_TARGETS:
        if _is_done(name, smote):
            print(f"  [{mode:8s}] {name:10s} SKIP (already checkpointed)")
            rows.append({"phase": mode, "dataset": name, "status": "SKIP", "seconds": None})
            continue
        t0 = time.time()
        try:
            result = _run_one(name, smote)
            secs = round(result.get("duration_seconds", 0) or (time.time()-t0), 1)
            rows.append({"phase": mode, "dataset": name, "status": "OK",
                         "best_model": result.get("best_model"),
                         "churn_rate": round(result["churn_rate"],4) if result.get("churn_rate") is not None else None,
                         "imbalance": round(result["imbalance_ratio"],2) if result.get("imbalance_ratio") is not None else None,
                         "seconds": secs})
            print(f"  [{mode:8s}] {name:10s} OK  best={result.get('best_model')}  {secs:6.1f}s")
            open(_marker(name, smote), "w").close()
            _backup_zip(name, smote)
        except Exception as exc:
            msg = str(exc)
            rows.append({"phase": mode, "dataset": name, "status": "FAILED", "seconds": round(time.time()-t0,1)})
            print(f"  [{mode:8s}] {name:10s} FAILED: {msg[:120]}")
            if any(k in msg for k in ("No such file", "FileNotFound", "does not exist")):
                print("    Raw multi-category data not attached under /kaggle/input/datasets/mkechinov/...")
summary = pd.DataFrame(rows)
display(summary)
print(f"\nREES46 sweep done in {(time.time()-sweep_start)/60:.1f} min.")
print("Download /kaggle/working — outputs are under results/rees46/{original,smote}/ and figures/rees46/...")


In [ ]:
# ── Inspect the produced REES46 artefacts ─────────────
import os
for d in ("results", "figures", "models", "processed_data"):
    p = os.path.join(PROJECT_ROOT, d, "rees46")
    print(f"\n{d}/rees46/")
    if not os.path.isdir(p):
        print("   (missing)")
        continue
    for mode in sorted(os.listdir(p)):
        mp = os.path.join(p, mode)
        n = sum(len(files) for _,_,files in os.walk(mp)) if os.path.isdir(mp) else 1
        print(f"   {mode}/  ({n} files)")

# Print the model metrics tables if present.
import pandas as pd
for mode in ("original", "smote"):
    mp = os.path.join(PROJECT_ROOT, "results", "rees46", mode, "model_metrics", f"model_metrics{mode}_smote.csv" if mode=="smote" else "model_metrics.csv")
    alt = os.path.join(PROJECT_ROOT, "results", "rees46", mode, "model_metrics")
    if os.path.isfile(mp):
        print(f"\n--- {mode} model metrics ---")
        t = pd.read_csv(mp)
        cols = [c for c in ("model","roc_auc","avg_precision","f1","accuracy","precision","recall","n_test") if c in t.columns]
        display(t[cols])
    else:
        d = alt if os.path.isdir(alt) else None
        if d:
            print(f"\n--- {mode} model_metrics files: {sorted(os.listdir(d))}")
        else:
            print(f"\n--- {mode} model_metrics: not found")


## After the run

1. Copy `/kaggle/working/results/rees46/`, `figures/rees46/` (and `models/rees46/`, `processed_data/rees46/`) back to the repo.
2. Keep the OLD contaminated outputs (`results/rees46/original|smote/...` from the ~1.0 native-label run) **archived separately** — do not overwrite or delete them; the new temporal outputs live alongside as a distinct audit trail.
3. Wrap up: write `src/report.md` REES46 temporal section with the new (non-1.0) churn rates and metrics, and report whatever the clean run produced — do not tune to hit any target.

### Audit note
The historical ~0.85 REES46 result was never recorded in this repo (exp-v2 holds only data_quality files; the only REES46 metrics were the ~1.0 native-label leakage). This run reports the clean temporal numbers as they are.
